# Generate Simulated Dynamic Multilayer Link Data

This notebook generates dynamic multilayer DGL heterographs for `DyMLH-Link`. Each year is a multilayer network with one node type and multiple edge types. The task is to use 2015-2019 snapshots to predict one target layer in 2020.


In [ ]:
from pathlib import Path
import numpy as np
import torch
import dgl

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

OUT_DIR = Path('data/simulated_multilayer')
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2015, 2021))
HISTORY_YEARS = YEARS[:-1]
TARGET_YEAR = YEARS[-1]
NODE_TYPE = 'node'
LAYER_NAMES = ['layer_0', 'layer_1', 'layer_2']
TARGET_LAYER = 'layer_0'

NUM_GLOBAL_NODES = 3000
NUM_COMMUNITIES = 8
FEAT_DIM = 64
LATENT_DIM = 32
MAX_RANDOM_PAIRS = 850_000
MAKE_UNDIRECTED = True

LAYER_CONFIG = {
    'layer_0': {'edge_bias': -2.0, 'same_comm_bonus': 2.5, 'previous_bonus': 2.8, 'recent_bonus': 1.1, 'keep_prob': 0.75},
    'layer_1': {'edge_bias': -2.2, 'same_comm_bonus': 1.8, 'previous_bonus': 2.1, 'recent_bonus': 0.8, 'keep_prob': 0.64},
    'layer_2': {'edge_bias': -2.4, 'same_comm_bonus': 1.2, 'previous_bonus': 1.6, 'recent_bonus': 0.6, 'keep_prob': 0.55},
}
TRAIN_RATIO, VALID_RATIO, TEST_RATIO = 0.7, 0.1, 0.2
print('Output directory:', OUT_DIR.resolve())


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

community = rng.integers(0, NUM_COMMUNITIES, size=NUM_GLOBAL_NODES, dtype=np.int64)
community_centers = rng.normal(0, 1.2, size=(NUM_COMMUNITIES, LATENT_DIM)).astype(np.float32)
node_offsets = rng.normal(0, 0.55, size=(NUM_GLOBAL_NODES, LATENT_DIM)).astype(np.float32)
layer_vectors = {layer: rng.normal(0, 0.45, size=(LATENT_DIM,)).astype(np.float32) for layer in LAYER_NAMES}
feature_projection = rng.normal(0, 0.45, size=(LATENT_DIM + NUM_COMMUNITIES, FEAT_DIM)).astype(np.float32)

def active_nodes_for_year(year_idx):
    active_prob = min(0.98, 0.88 + 0.015 * year_idx)
    active = rng.random(NUM_GLOBAL_NODES) < active_prob
    active[: int(NUM_GLOBAL_NODES * 0.55)] = True
    return np.flatnonzero(active).astype(np.int64)

def latent_for_year(global_ids, year_idx):
    drift = 0.05 * year_idx * rng.normal(0, 1, size=(NUM_COMMUNITIES, LATENT_DIM)).astype(np.float32)
    centers = community_centers[community[global_ids]] + drift[community[global_ids]]
    noise = rng.normal(0, 0.12, size=(len(global_ids), LATENT_DIM)).astype(np.float32)
    return centers + node_offsets[global_ids] + noise

def features_from_latent(global_ids, latent):
    one_hot = np.eye(NUM_COMMUNITIES, dtype=np.float32)[community[global_ids]]
    raw = np.concatenate([latent, one_hot], axis=1)
    return raw @ feature_projection + rng.normal(0, 0.03, size=(len(global_ids), FEAT_DIM)).astype(np.float32)

def unique_pairs(src, dst):
    keep = src != dst
    src, dst = src[keep], dst[keep]
    if len(src) == 0:
        return src.astype(np.int64), dst.astype(np.int64)
    pairs = np.unique(np.stack([src, dst], axis=1), axis=0)
    return pairs[:, 0].astype(np.int64), pairs[:, 1].astype(np.int64)

def localize_edges(edge_set, global_to_local):
    src, dst = [], []
    for u, v in edge_set:
        if u in global_to_local and v in global_to_local:
            src.append(global_to_local[u]); dst.append(global_to_local[v])
    if not src:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64)
    return np.asarray(src, dtype=np.int64), np.asarray(dst, dtype=np.int64)

def sample_layer_edges(global_ids, latent, layer_name, previous_edges, recent_edges):
    n = len(global_ids)
    cfg = LAYER_CONFIG[layer_name]
    global_to_local = {int(gid): idx for idx, gid in enumerate(global_ids.tolist())}
    rand_src = rng.integers(0, n, size=MAX_RANDOM_PAIRS, dtype=np.int64)
    rand_dst = rng.integers(0, n, size=MAX_RANDOM_PAIRS, dtype=np.int64)
    prev_src, prev_dst = localize_edges(previous_edges, global_to_local)
    if len(prev_src) > 0:
        keep = rng.random(len(prev_src)) < cfg['keep_prob']
        prev_src, prev_dst = prev_src[keep], prev_dst[keep]
    src, dst = unique_pairs(np.concatenate([rand_src, prev_src]), np.concatenate([rand_dst, prev_dst]))
    src_gid, dst_gid = global_ids[src], global_ids[dst]
    same_comm = (community[src_gid] == community[dst_gid]).astype(np.float32)
    latent_score = np.sum((latent[src] + layer_vectors[layer_name]) * latent[dst], axis=1) / np.sqrt(latent.shape[1])
    prev_flag = np.array([(int(u), int(v)) in previous_edges for u, v in zip(src_gid, dst_gid)], dtype=np.float32)
    recent_flag = np.array([(int(u), int(v)) in recent_edges for u, v in zip(src_gid, dst_gid)], dtype=np.float32)
    score = cfg['edge_bias'] + latent_score + cfg['same_comm_bonus'] * same_comm + cfg['previous_bonus'] * prev_flag + cfg['recent_bonus'] * recent_flag
    chosen = rng.random(len(score)) < sigmoid(score)
    src, dst = unique_pairs(src[chosen], dst[chosen])
    if MAKE_UNDIRECTED and len(src) > 0:
        src, dst = unique_pairs(np.concatenate([src, dst]), np.concatenate([dst, src]))
    return src, dst, set(zip(global_ids[src].tolist(), global_ids[dst].tolist()))

def make_masks(num_edges):
    order = rng.permutation(num_edges)
    n_train = int(num_edges * TRAIN_RATIO); n_valid = int(num_edges * VALID_RATIO)
    train_idx, valid_idx, test_idx = order[:n_train], order[n_train:n_train + n_valid], order[n_train + n_valid:]
    train_mask = torch.zeros(num_edges, dtype=torch.bool); valid_mask = torch.zeros(num_edges, dtype=torch.bool); test_mask = torch.zeros(num_edges, dtype=torch.bool)
    train_mask[torch.as_tensor(train_idx)] = True; valid_mask[torch.as_tensor(valid_idx)] = True; test_mask[torch.as_tensor(test_idx)] = True
    return train_mask, valid_mask, test_mask


In [ ]:
previous_edges_by_layer = {layer: set() for layer in LAYER_NAMES}
recent_edges_by_layer = {layer: set() for layer in LAYER_NAMES}
year_edge_sets = {}

for year_idx, year in enumerate(YEARS):
    global_ids = active_nodes_for_year(year_idx)
    latent = latent_for_year(global_ids, year_idx)
    features = features_from_latent(global_ids, latent)
    graph_data = {}
    current_edges = {}
    for layer in LAYER_NAMES:
        src, dst, edge_set = sample_layer_edges(global_ids, latent, layer, previous_edges_by_layer[layer], recent_edges_by_layer[layer])
        graph_data[(NODE_TYPE, layer, NODE_TYPE)] = (torch.from_numpy(src), torch.from_numpy(dst))
        current_edges[layer] = edge_set
    g = dgl.heterograph(graph_data, num_nodes_dict={NODE_TYPE: len(global_ids)})
    g.nodes[NODE_TYPE].data['feat'] = torch.tensor(features, dtype=torch.float32)
    g.nodes[NODE_TYPE].data['global_id'] = torch.tensor(global_ids, dtype=torch.long)
    g.nodes[NODE_TYPE].data['community'] = torch.tensor(community[global_ids], dtype=torch.long)
    if year == TARGET_YEAR:
        target_etype = (NODE_TYPE, TARGET_LAYER, NODE_TYPE)
        train_mask, valid_mask, test_mask = make_masks(g.num_edges(target_etype))
        g.edges[target_etype].data['train_mask'] = train_mask
        g.edges[target_etype].data['valid_mask'] = valid_mask
        g.edges[target_etype].data['test_mask'] = test_mask
    path = OUT_DIR / f'graph_{year}.bin'
    dgl.save_graphs(str(path), [g], {'year': torch.tensor([year])})
    year_edge_sets[year] = current_edges
    print(year, g, 'saved to', path)
    for layer in LAYER_NAMES:
        print(' ', layer, 'edges:', g.num_edges((NODE_TYPE, layer, NODE_TYPE)))
        recent_edges_by_layer[layer] = current_edges[layer]
        previous_edges_by_layer[layer] |= current_edges[layer]


In [ ]:
target_edges = year_edge_sets[TARGET_YEAR][TARGET_LAYER]
history_edges = set().union(*(year_edge_sets[y][TARGET_LAYER] for y in HISTORY_YEARS))
overlap = len(target_edges & history_edges)
print('target layer:', TARGET_LAYER)
print('target-year target-layer edges:', len(target_edges))
print('historical target-layer unique edges:', len(history_edges))
print('target-layer edges previously observed:', overlap)
print('overlap ratio:', overlap / max(1, len(target_edges)))
graphs, metadata = dgl.load_graphs(str(OUT_DIR / f'graph_{TARGET_YEAR}.bin'))
g = graphs[0]
print(g)
print('node data:', list(g.nodes[NODE_TYPE].data.keys()))
print('target edge data:', list(g.edges[(NODE_TYPE, TARGET_LAYER, NODE_TYPE)].data.keys()))


In [ ]:
snapshot_bins = ','.join(str(OUT_DIR / f'graph_{year}.bin') for year in HISTORY_YEARS)
target_bin = str(OUT_DIR / f'graph_{TARGET_YEAR}.bin')
use_layers = ','.join(LAYER_NAMES)
undirected_flag = ' \\n  --undirected' if MAKE_UNDIRECTED else ''
cmd = f'''python -u Link_Prediction.py \\n  --snapshot-bins {snapshot_bins} \\n  --target-bin {target_bin} \\n  --node-type {NODE_TYPE} \\n  --target-layer {TARGET_LAYER} \\n  --use-layers {use_layers} \\n  --feat-key feat \\n  --global-id-key global_id \\n  --hidden-dim 128 \\n  --gnn-layers 2 \\n  --sage-aggregator-type mean \\n  --layer-fusion attention \\n  --temporal-model gru \\n  --temporal-layers 1 \\n  --predictor dot \\n  --negative-ratio 1.0 \\n  --eval-negative-ratio 1.0 \\n  --negative-exclude-layers target \\n  --epochs 200 \\n  --patience 30 \\n  --early-stop-metric auc \\n  --log-every 10 \\n  --output-dir outputs{undirected_flag}'''
print(cmd)
